# Short-List Promising Models & Fine-Tune the System

In [1]:
import pickle
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from custom_transformers import *
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import cross_validate
from sklearn.model_selection import RandomizedSearchCV
from functools import partial
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.base import clone
from sklearn.inspection import permutation_importance

In [2]:
train_set = pd.read_parquet("train_set.parquet")
y = train_set["diseased"]
X = train_set.drop(columns="diseased")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [3]:
log_reg = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])
log_reg_recall = cross_val_score(log_reg, X, y, cv=cv, scoring="recall")
log_reg_precision = cross_val_score(log_reg, X, y, cv=cv, scoring="precision")
log_reg_f1 = cross_val_score(log_reg, X, y, cv=cv, scoring="f1")

pd.DataFrame({
    "recall": log_reg_recall,
    "precision": log_reg_precision,
    "f1": log_reg_f1,
}).mean().round(3)

recall       0.749
precision    0.428
f1           0.543
dtype: float64

Logistic regression performed well. Across the 5 folds it found about 75% of the sick sightings with a recall of 0.749, and about 43% of the sightings it flagged were actually sick with a precision of 0.428. The F1 score was 0.543. Recall was a better score than precision and f1.

In [4]:
sgd = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", SGDClassifier(loss="log_loss", class_weight="balanced", random_state=42)),
])
sgd_recall = cross_val_score(sgd, X, y, cv=cv, scoring="recall")
sgd_precision = cross_val_score(sgd, X, y, cv=cv, scoring="precision")
sgd_f1 = cross_val_score(sgd, X, y, cv=cv, scoring="f1")

pd.DataFrame({
    "recall": sgd_recall,
    "precision": sgd_precision,
    "f1": sgd_f1,
}).mean().round(3)

recall       0.785
precision    0.303
f1           0.425
dtype: float64

The SGD classifier had the highest recall of the six models at 0.785, but its precision dropped to 0.303, so most of the sightings it flagged were healthy. Its F1 score was 0.425.

In [5]:
svc = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", SVC(class_weight="balanced", random_state=42)),
])
svc_recall = cross_val_score(svc, X, y, cv=cv, scoring="recall")
svc_precision = cross_val_score(svc, X, y, cv=cv, scoring="precision")
svc_f1 = cross_val_score(svc, X, y, cv=cv, scoring="f1")

pd.DataFrame({
    "recall": svc_recall,
    "precision": svc_precision,
    "f1": svc_f1,
}).mean().round(3)

recall       0.716
precision    0.448
f1           0.551
dtype: float64

The SVC performed well and was very close to logistic regression. It found about 72% of the sick sightings with a recall of 0.716 and had a precision of 0.448, giving an F1 score of 0.551.

In [6]:
tree = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", DecisionTreeClassifier(class_weight="balanced", random_state=42)),
])
tree_recall = cross_val_score(tree, X, y, cv=cv, scoring="recall")
tree_precision = cross_val_score(tree, X, y, cv=cv, scoring="precision")
tree_f1 = cross_val_score(tree, X, y, cv=cv, scoring="f1")

pd.DataFrame({
    "recall": tree_recall,
    "precision": tree_precision,
    "f1": tree_f1,
}).mean().round(3)

recall       0.442
precision    0.438
f1           0.440
dtype: float64

The decision tree was the weakest model. It found only 44% of the sick sightings with a recall of 0.442, the lowest of the six, and had a precision of 0.438 and an F1 score of 0.440. A single tree with no depth limit overfits the training folds.

In [7]:
forest = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", RandomForestClassifier(class_weight="balanced", random_state=42)),
])
forest_recall = cross_val_score(forest, X, y, cv=cv, scoring="recall")
forest_precision = cross_val_score(forest, X, y, cv=cv, scoring="precision")
forest_f1 = cross_val_score(forest, X, y, cv=cv, scoring="f1")

pd.DataFrame({
    "recall": forest_recall,
    "precision": forest_precision,
    "f1": forest_f1,
}).mean().round(3)

recall       0.617
precision    0.546
f1           0.576
dtype: float64

The random forest performed well and had the highest F1 score of the six models at 0.576. Its precision of 0.546 was better than the linear models, but it found fewer sick sightings with a recall of 0.617.

In [8]:
boost = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", GradientBoostingClassifier(random_state=42)),
])
boost_recall = cross_val_score(boost, X, y, cv=cv, scoring="recall")
boost_precision = cross_val_score(boost, X, y, cv=cv, scoring="precision")
boost_f1 = cross_val_score(boost, X, y, cv=cv, scoring="f1")

pd.DataFrame({
    "recall": boost_recall,
    "precision": boost_precision,
    "f1": boost_f1,
}).mean().round(3)

recall       0.466
precision    0.679
f1           0.549
dtype: float64

Gradient boosting had the highest precision at 0.679, so most of the sightings it flagged were actually sick. Its recall was lower at 0.466 because it does not support class weights, so it leans toward predicting healthy. Its F1 score was 0.549, close to the SVC.

In [9]:
recall_by_fold = pd.DataFrame({
    "Logistic Regression": log_reg_recall,
    "SGD": sgd_recall,
    "SVC": svc_recall,
    "Decision Tree": tree_recall,
    "Random Forest": forest_recall,
    "Gradient Boosting": boost_recall
})

recall_by_fold.agg(["mean", "std"]).T.round(3)

,mean,std
Logistic Regression,0.749,0.029
SGD,0.785,0.137
SVC,0.716,0.044
Decision Tree,0.442,0.045
Random Forest,0.617,0.063
Gradient Boosting,0.466,0.061


In [10]:
models = {
    "Logistic Regression": log_reg,
    "SGD": sgd,
    "SVC": svc,
    "Decision Tree": tree,
    "Random Forest": forest,
    "Gradient Boosting": boost
}

train_idx, valid_idx = next(cv.split(X, y))
prep = make_preprocessor()
X_fold_train = prep.fit_transform(X.iloc[train_idx])
X_fold_valid = prep.transform(X.iloc[valid_idx])

top_features = {}

validation_predictions = pd.DataFrame(index=y.iloc[valid_idx].index)

for name, pipeline in models.items():
    model = clone(pipeline.named_steps["model"])
    model.fit(X_fold_train, y.iloc[train_idx])
    validation_predictions[name] = model.predict(X_fold_valid)

    result = permutation_importance(
        model, X_fold_valid, y.iloc[valid_idx],
        scoring="recall", n_repeats=3, random_state=42
    )
    importance = pd.Series(result.importances_mean, index=X_fold_valid.columns)
    top_features[name] = importance.nlargest(3)

pd.DataFrame(top_features).T.round(3)

,Location_Above Ground,Primary Fur Color_Black,Primary Fur Color_Cinnamon,Primary Fur Color_Gray,Runs from,Y,log_height,log_precip_mm,prev_tmin_c
Logistic Regression,NaN,NaN,0.109,0.339,NaN,NaN,NaN,NaN,0.044
SGD,0.027,NaN,NaN,0.290,NaN,NaN,0.033,NaN,NaN
SVC,NaN,0.066,0.087,0.454,NaN,NaN,NaN,NaN,NaN
Decision Tree,NaN,NaN,NaN,0.372,NaN,0.098,NaN,0.098,NaN
Random Forest,NaN,0.066,0.126,0.284,NaN,NaN,NaN,NaN,NaN
Gradient Boosting,NaN,0.044,NaN,0.262,0.044,NaN,NaN,NaN,NaN


In [11]:
actual = y.iloc[valid_idx]

pd.DataFrame({
    "Missed sick sightings": (validation_predictions.loc[actual] == False).sum(),
    "False alarms": (validation_predictions.loc[~actual] == True).sum()
})

,Missed sick sightings,False alarms
Logistic Regression,16,47
SGD,16,164
SVC,21,49
Decision Tree,33,28
Random Forest,27,20
Gradient Boosting,38,7


In [12]:
rf_pred = validation_predictions["Random Forest"]

missed_sick = actual & ~rf_pred
false_alarm = ~actual & rf_pred

review_columns = [
    "Primary Fur Color", "Approaches", "Color notes",
    "Specific Location", "Other Activities", "Other Interactions"
]
reports = X.loc[actual.index, review_columns]

examples = pd.concat([
    reports.loc[missed_sick].head(1).assign(Error="Missed sick sighting"),
    reports.loc[false_alarm].head(1).assign(Error="False alarm")
])

with pd.option_context("display.max_colwidth", None):
    display(examples.set_index("Error").T)

Error,Missed sick sighting,False alarm
Primary Fur Color,Gray,Cinnamon
Approaches,True,False
Color notes,NaN,NaN
Specific Location,NaN,on tree trunk
Other Activities,NaN,NaN
Other Interactions,NaN,NaN


## Model selection for tuning

We selected SGD for its recall (0.785) and Random Forest for its F1 score (0.576). Each had the highest score on that metric among the six baseline models. We also selected SVC because it combined relatively high recall (0.716) with an F1 score of 0.551. For the additional ensemble, we chose Gradient Boosting. It had the highest precision (0.679), and we want to see whether tuning can improve its recall (0.466) while keeping false alarms low.

In [13]:
def recall_at_45(model, X_valid, y_valid, limit=45):
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X_valid)
    else:
        scores = model.predict_proba(X_valid)[:, 1]

    reports = pd.DataFrame({
        "date": X_valid["Date"],
        "score": scores,
        "diseased": y_valid
    })

    ranked = reports.sort_values("score", ascending=False, kind="stable")
    selected = ranked.groupby("date").head(limit)

    return selected["diseased"].sum() / reports["diseased"].sum()

We keep all training sightings from the same date in one fold so we can evaluate a full day’s ranking together. For each validation date, we rank the reports by model score and select up to 45. We then measure what fraction of sick-labeled sightings in the fold were selected.

In [14]:
date_cv = GroupKFold(n_splits=5)
tuning_folds = list(date_cv.split(X, y, groups=X["Date"]))

for fold, (_, valid_idx) in enumerate(tuning_folds, start=1):
    dates = X.iloc[valid_idx]["Date"].dt.strftime("%b %d").unique()
    print(f"Fold {fold}: {', '.join(sorted(dates))}")

Fold 1: Oct 13, Oct 19
Fold 2: Oct 07, Oct 18
Fold 3: Oct 12, Oct 14
Fold 4: Oct 06, Oct 17, Oct 20
Fold 5: Oct 08, Oct 10


In [15]:
scoring = {
    "recall_at_45": recall_at_45,
    "recall": "recall",
    "precision": "precision",
    "f1": "f1"
}

sgd_baseline = cross_validate(
    sgd, X, y,
    cv=tuning_folds,
    scoring=scoring,
    error_score="raise"
)

pd.DataFrame({
    metric: sgd_baseline[f"test_{metric}"]
    for metric in scoring
}).mean().round(3)

recall_at_45    0.669
recall          0.760
precision       0.366
f1              0.471
dtype: float64

In [16]:
sgd_params = {
    "model__loss": ["log_loss", "hinge", "modified_huber"],
    "model__alpha": [0.00001, 0.0001, 0.001, 0.01, 0.1],
    "model__penalty": ["l2", "l1", "elasticnet"],
    "model__class_weight": [None, "balanced"],
    "model__average": [False, True],
    "model__fit_intercept": [False, True],

    "preprocessor__columns__num__impute__strategy": ["median", "mean"],
    "preprocessor__columns__cat__impute__strategy": ["constant", "most_frequent"],
    "preprocessor__columns__weather": [numeric_steps(), "drop"],
    "preprocessor__columns__events": [numeric_steps(), "drop"]
}

We used randomized search because testing every combination of our 10 hyperparameters would require thousands of fits per model. For each model, we sampled 30 combinations and selected the one with the highest average recall at a daily limit of 45 reports across the same five validation folds.

In [17]:
sgd_search = RandomizedSearchCV(
    estimator=sgd,
    param_distributions=sgd_params,
    n_iter=30,
    scoring=scoring,
    refit="recall_at_45",
    cv=tuning_folds,
    random_state=42,
    error_score="raise"
)

sgd_search.fit(X, y)

best = sgd_search.best_index_

pd.DataFrame({
    "Baseline": {
        metric: sgd_baseline[f"test_{metric}"].mean()
        for metric in scoring
    },
    "Tuned SGD": {
        metric: sgd_search.cv_results_[f"mean_test_{metric}"][best]
        for metric in scoring
    }
}).round(3)

/Users/ranayum/Courses/294/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/ranayum/Courses/294/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/ranayum/Courses/294/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape

,Baseline,Tuned SGD
recall_at_45,0.669,0.710
recall,0.760,0.464
precision,0.366,0.684
f1,0.471,0.527


With a daily limit of 45 reports, tuned SGD improved mean validation recall from 0.669 to 0.710. At the default decision threshold, precision increased from 0.366 to 0.684 and recall fell from 0.760 to 0.464. F1 improved from 0.471 to 0.527.

In [18]:
sgd_search.best_params_

{'preprocessor__columns__weather': 'drop',
 'preprocessor__columns__num__impute__strategy': 'median',
 'preprocessor__columns__events': Pipeline(steps=[('impute', SimpleImputer(strategy='median')),
                 ('scale', StandardScaler())]),
 'preprocessor__columns__cat__impute__strategy': 'constant',
 'model__penalty': 'l2',
 'model__loss': 'hinge',
 'model__fit_intercept': True,
 'model__class_weight': None,
 'model__average': True,
 'model__alpha': 1e-05}

In [19]:
forest_baseline = cross_validate(
    forest, X, y,
    cv=tuning_folds,
    scoring=scoring,
    error_score="raise"
)

pd.Series({
    metric: forest_baseline[f"test_{metric}"].mean()
    for metric in scoring
}).round(3)

recall_at_45    0.713
recall          0.595
precision       0.596
f1              0.574
dtype: float64

In [20]:
forest_params = {
    "model__n_estimators": [100, 200, 400],
    "model__max_depth": [3, 6, 12, None],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 5, 10],
    "model__max_features": ["sqrt", 0.5, 1.0],
    "model__class_weight": [None, "balanced", "balanced_subsample"],

    "preprocessor__columns__num__impute__strategy": ["median", "mean"],
    "preprocessor__columns__cat__impute__strategy": ["constant", "most_frequent"],
    "preprocessor__columns__weather": [numeric_steps(), "drop"],
    "preprocessor__columns__events": [numeric_steps(), "drop"]
}

In [21]:
forest_search = RandomizedSearchCV(
    estimator=forest,
    param_distributions=forest_params,
    n_iter=30,
    scoring=scoring,
    refit="recall_at_45",
    cv=tuning_folds,
    random_state=42,
    error_score="raise"
)

forest_search.fit(X, y)
best = forest_search.best_index_

pd.DataFrame({
    "Baseline": {
        metric: forest_baseline[f"test_{metric}"].mean()
        for metric in scoring
    },
    "Tuned Forest": {
        metric: forest_search.cv_results_[f"mean_test_{metric}"][best]
        for metric in scoring
    }
}).round(3)

,Baseline,Tuned Forest
recall_at_45,0.713,0.715
recall,0.595,0.685
precision,0.596,0.554
f1,0.574,0.595


With a daily limit of 45 reports, mean validation recall changed only slightly, from 0.713 to 0.715. At the default decision threshold, recall increased from 0.595 to 0.685 while precision fell from 0.596 to 0.554. F1 improved from 0.574 to 0.595.

In [22]:
forest_search.best_params_

{'preprocessor__columns__weather': 'drop',
 'preprocessor__columns__num__impute__strategy': 'median',
 'preprocessor__columns__events': 'drop',
 'preprocessor__columns__cat__impute__strategy': 'constant',
 'model__n_estimators': 100,
 'model__min_samples_split': 10,
 'model__min_samples_leaf': 2,
 'model__max_features': 'sqrt',
 'model__max_depth': 12,
 'model__class_weight': 'balanced_subsample'}

We tested whether to include weather and park event features during tuning. Among the Random Forest settings we tried, the one with the highest mean validation recall at 45 reports per day excluded both groups. We used that setting for the final model.

In [23]:
svc_baseline = cross_validate(
    svc, X, y,
    cv=tuning_folds,
    scoring=scoring,
    error_score="raise"
)

pd.Series({
    metric: svc_baseline[f"test_{metric}"].mean()
    for metric in scoring
}).round(3)

recall_at_45    0.701
recall          0.725
precision       0.492
f1              0.573
dtype: float64

In [24]:
svc_params = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__gamma": ["scale", "auto", 0.001, 0.01, 0.1],
    "model__kernel": ["rbf", "sigmoid"],
    "model__class_weight": [None, "balanced"],

    "preprocessor__columns__num__impute__strategy": ["median", "mean"],
    "preprocessor__columns__num__scale": [StandardScaler(), "passthrough"],
    "preprocessor__columns__cat__impute__strategy": ["constant", "most_frequent"],
    "preprocessor__columns__cat__onehot__min_frequency": [None, 5, 20],
    "preprocessor__columns__weather": [numeric_steps(), "drop"],
    "preprocessor__columns__events": [numeric_steps(), "drop"]
}

In [25]:
svc_search = RandomizedSearchCV(
    estimator=svc,
    param_distributions=svc_params,
    n_iter=30,
    scoring=scoring,
    refit="recall_at_45",
    cv=tuning_folds,
    random_state=42,
    error_score="raise"
)

svc_search.fit(X, y)
best = svc_search.best_index_

pd.DataFrame({
    "Baseline": {
        metric: svc_baseline[f"test_{metric}"].mean()
        for metric in scoring
    },
    "Tuned SVC": {
        metric: svc_search.cv_results_[f"mean_test_{metric}"][best]
        for metric in scoring
    }
}).round(3)

/Users/ranayum/Courses/294/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/ranayum/Courses/294/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/ranayum/Courses/294/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape

,Baseline,Tuned SVC
recall_at_45,0.701,0.703
recall,0.725,0.724
precision,0.492,0.504
f1,0.573,0.577


With a daily limit of 45 reports, mean validation recall changed from 0.701 to 0.703. At the default threshold, recall stayed nearly the same (0.725 to 0.724), while precision rose from 0.492 to 0.504 and F1 rose from 0.573 to 0.577.

In [26]:
boost_baseline = cross_validate(
    boost, X, y,
    cv=tuning_folds,
    scoring=scoring,
    error_score="raise"
)

pd.Series({
    metric: boost_baseline[f"test_{metric}"].mean()
    for metric in scoring
}).round(3)

recall_at_45    0.672
recall          0.397
precision       0.536
f1              0.430
dtype: float64

In [27]:
boost_params = {
    "model__n_estimators": [50, 100, 200, 400],
    "model__learning_rate": [0.01, 0.03, 0.1, 0.2],
    "model__max_depth": [1, 2, 3, 5],
    "model__min_samples_leaf": [1, 3, 5, 10],
    "model__subsample": [0.5, 0.75, 1.0],
    "model__max_features": [None, "sqrt", 0.5],

    "preprocessor__columns__num__impute__strategy": ["median", "mean"],
    "preprocessor__columns__cat__impute__strategy": ["constant", "most_frequent"],
    "preprocessor__columns__weather": [numeric_steps(), "drop"],
    "preprocessor__columns__events": [numeric_steps(), "drop"]
}

In [28]:
boost_search = RandomizedSearchCV(
    estimator=boost,
    param_distributions=boost_params,
    n_iter=30,
    scoring=scoring,
    refit="recall_at_45",
    cv=tuning_folds,
    random_state=42,
    error_score="raise"
)

boost_search.fit(X, y)
best = boost_search.best_index_

pd.DataFrame({
    "Baseline": {
        metric: boost_baseline[f"test_{metric}"].mean()
        for metric in scoring
    },
    "Tuned Boost": {
        metric: boost_search.cv_results_[f"mean_test_{metric}"][best]
        for metric in scoring
    }
}).round(3)

/Users/ranayum/Courses/294/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/ranayum/Courses/294/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/ranayum/Courses/294/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape

,Baseline,Tuned Boost
recall_at_45,0.672,0.717
recall,0.397,0.426
precision,0.536,0.648
f1,0.430,0.489


With a daily limit of 45 reports, tuning improved mean validation recall from 0.672 to 0.717. At the default threshold, precision rose from 0.536 to 0.648, recall from 0.397 to 0.426, and F1 from 0.430 to 0.489. Gradient Boosting had the highest recall at 45, narrowly ahead of Random Forest (0.715), while Random Forest had the highest F1 (0.595).

In [29]:
searches = {
    "SGD": sgd_search,
    "Random Forest": forest_search,
    "SVC": svc_search,
    "Gradient Boosting": boost_search
}

comparison = pd.DataFrame({
    name: {
        metric: search.cv_results_[f"mean_test_{metric}"][search.best_index_]
        for metric in scoring
    }
    for name, search in searches.items()
}).T

comparison.sort_values("recall_at_45", ascending=False).round(3)

,recall_at_45,recall,precision,f1
Gradient Boosting,0.717,0.426,0.648,0.489
Random Forest,0.715,0.685,0.554,0.595
SGD,0.710,0.464,0.684,0.527
SVC,0.703,0.724,0.504,0.577


In [30]:
limits = [40, 45, 50]
capacity_scoring = {
    f"recall_at_{limit}": partial(recall_at_45, limit=limit)
    for limit in limits
}

capacity_results = {}

for name, search in searches.items():
    scores = cross_validate(
        search.best_estimator_, X, y,
        cv=tuning_folds,
        scoring=capacity_scoring,
        error_score="raise"
    )
    capacity_results[name] = {
        metric: scores[f"test_{metric}"].mean()
        for metric in capacity_scoring
    }

pd.DataFrame(capacity_results).T.sort_values(
    "recall_at_45", ascending=False
).round(3)

,recall_at_40,recall_at_45,recall_at_50
Gradient Boosting,0.670,0.717,0.732
Random Forest,0.679,0.715,0.749
SGD,0.660,0.710,0.732
SVC,0.659,0.703,0.726


## Final model selection

We selected Random Forest because the team can investigate 40–50 reports per day. It had the highest validation recall at 40 and 50 reports and was close to Gradient Boosting at 45 (0.715 versus 0.717). It also had the highest F1 score at the default threshold (0.595).

In [31]:
final_model = forest_search.best_estimator_

with open("model.pkl", "wb") as file:
    pickle.dump(final_model, file)

In [32]:
test_set = pd.read_parquet("test_set.parquet")
X_test = test_set.drop(columns="diseased")
y_test = test_set["diseased"]

y_pred = final_model.predict(X_test)

print(classification_report(
    y_test, y_pred,
    labels=[False, True],
    target_names=["Healthy", "Sick"],
    digits=3
))

pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=[False, True]),
    index=["Actual healthy", "Actual sick"],
    columns=["Predicted healthy", "Predicted sick"]
)

              precision    recall  f1-score   support

     Healthy      0.932     0.930     0.931       529
        Sick      0.519     0.526     0.523        76

    accuracy                          0.879       605
   macro avg      0.726     0.728     0.727       605
weighted avg      0.880     0.879     0.880       605



,Predicted healthy,Predicted sick
Actual healthy,492,37
Actual sick,36,40


At the default threshold, Random Forest identified 40 of 76 sick-labeled test sightings and missed 36. Of the 77 sightings it flagged, 40 were labeled sick, giving a precision of 0.519. Recall fell from 0.685 in validation to 0.526 on the test set, and F1 fell from 0.595 to 0.523.

In [33]:
reports = test_set[["Date", "diseased"]].copy()
reports["score"] = final_model.predict_proba(X_test)[:, 1]
reports = reports.sort_values("score", ascending=False, kind="stable")
reports["daily_rank"] = reports.groupby("Date").cumcount() + 1

all_dates = pd.concat([X["Date"], X_test["Date"]])
test_share = X_test["Date"].value_counts() / all_dates.value_counts()

results = []
for limit in [40, 45, 50]:
    daily_quota = (limit * test_share).round()
    selected = reports[reports["daily_rank"] <= reports["Date"].map(daily_quota)]
    caught = selected["diseased"].sum()

    results.append({
        "Daily capacity": limit,
        "Selected": len(selected),
        "Sick sightings caught": caught,
        "Recall": caught / y_test.sum(),
        "Precision": caught / len(selected)
    })

pd.DataFrame(results).round(3)

,Daily capacity,Selected,Sick sightings caught,Recall,Precision
0,40,86,40,0.526,0.465
1,45,98,44,0.579,0.449
2,50,108,48,0.632,0.444


We scaled each day’s capture limit to the share of that day’s reports in the test set. At a daily capacity of 45, Random Forest selected 98 reports across the test period and identified 44 of 76 sick sightings (57.9% recall). Raising capacity to 50 selected 10 additional reports and identified 4 more sick sightings. Precision changed from 44.9% to 44.4%.